# bridgepandas — introduction

This notebook walks through the core features of `bridgepandas`: generating random deals, querying hand properties, and computing double-dummy scores.

In [ ]:
import bridgepandas as bp
import pandas as pd

## 1. Generating deals

`random_deals(n)` returns a DataFrame with four `BridgeHandArray` columns — one per direction.

In [ ]:
df = bp.random_deals(10, seed=42)
df

## 2. Hand accessors

All columns support pandas-style accessors for common bridge properties.

In [ ]:
df['north'].hcp

In [ ]:
# Suit lengths
pd.DataFrame({
    'S': df['north'].spades,
    'H': df['north'].hearts,
    'D': df['north'].diamonds,
    'C': df['north'].clubs,
})

In [ ]:
# Other hand statistics
pd.DataFrame({
    'hcp':         df['north'].hcp,
    'controls':    df['north'].controls,
    'losers':      df['north'].losers,
    'quick_tricks':df['north'].quick_tricks,
    'voids':       df['north'].voids,
    'singletons':  df['north'].singletons,
})

## 3. Constrained deals

Pass partial hand strings or `HandSet` constraints to `random_deals`.

In [ ]:
# North holds at least AK of spades
df_ak = bp.random_deals(5, north='AK/-/-/-', seed=1)
print(df_ak['north'].spades.tolist())   # all >= 2
print(df_ak['north'].has('SA').tolist())  # all True
df_ak

In [ ]:
# Use a HandSet for richer constraints (e.g. exactly 5 spades and 16+ HCP)
hs = (bp.h.SPADES == 5) & (bp.h.HCP >= 16) & (bp.h.HCP <= 40)
df_hs = bp.random_deals(5, north=hs, seed=7)
pd.DataFrame({'spades': df_hs['north'].spades, 'hcp': df_hs['north'].hcp})

### Counting hands with a HandSet

`HandSet.count()` returns the exact number of 13-card hands satisfying a constraint — no simulation needed.

In [ ]:
# HandSets count hands exactly — no simulation needed
nt_opener = (
    (bp.h.HCP >= 15) & (bp.h.HCP <= 17)
    & bp.h.SHAPE('any 4333 + any 5332 + any 4432')
)
nt_with_ha = nt_opener & bp.h.CARD('HA')

total   = nt_opener.count()
with_ha = nt_with_ha.count()
print(f'1NT openers:       {total:,}')
print(f'  ...holding ♥A:   {with_ha:,}  ({with_ha/total:.1%})')

## 4. Scalar Hand objects

Individual rows return `Hand` objects — int subclasses that print as hand strings and expose the same properties directly.

In [ ]:
hand = df.iloc[0].north
print(hand)
print(f'HCP={hand.hcp}  losers={hand.losers}  quick_tricks={hand.quick_tricks}')
print(f'Spades={hand.spades}  Hearts={hand.hearts}')
print(f'Has SA: {hand.has("SA")}  Has HK: {hand.has("HK")}')

## 5. Deal objects

`Deal` wraps all four hands and renders as a compass diagram.

In [ ]:
deal = bp.Deal(df.iloc[0])
print(deal)

## 6. Scoring

`score(contract, tricks, vulnerable)` returns the declarer's score.  
`score_ns(declared_contract, tricks, vuln)` returns the score from North-South's perspective.

In [ ]:
print(bp.score('3N', 9, False))   # NV 3NT made
print(bp.score('4S', 10, True))   # Vul 4S made
print(bp.score('3N', 8, False))   # NV 3NT down 1

In [ ]:
from bridgepandas.scoring import score_ns

print(score_ns('4S-N', 10, '-'))   # N declares 4S, made, NV  → +420
print(score_ns('4S-E', 10, 'ns'))  # E declares 4S, made, NS vul → -420 from NS view
print(score_ns('3N-N', 8, 'b'))    # N declares 3NT, down 1, both vul → -100

## 7. Double-dummy solving

`solve(df, trump, leader)` returns the number of tricks the leading side takes.  
`add_dds(df, contract, col_name, vuln)` computes tricks and scores in one step, caching results in a `_dds` column.

In [ ]:
df = bp.random_deals(20, seed=42)

# How many tricks can NS take in NT with North on lead?
ns_nt_tricks = 13 - bp.solve(df, trump='NT', leader='N')
ns_nt_tricks

In [ ]:
# Score a 3NT-N contract (NV) for every deal
bp.add_dds(df, '3N-N', 'score_3nn', '-')

# Score 4S-N (NV) — reuses cached NT tricks for declarer tricks
# then solves fresh for Spades
bp.add_dds(df, '4S-N', 'score_4sn', '-')

# Second call with same contract is instant (fully cached)
bp.add_dds(df, '3N-N', 'score_3nn_vul', 'ns')

df[['score_3nn', 'score_4sn', 'score_3nn_vul', '_dds']].head(10)

In [ ]:
# Per-row contracts
contracts = pd.Series(['3N-N', '4S-W', '3N-N', '4H-E', '2C-S'] * 4)
bp.add_dds(df, contracts, 'score_mixed', '-')
df[['score_mixed']].head(10)

## 8. Board numbers, dealer, and vulnerability

`board_number_to_dealer_vuln` and `dealer_vuln_to_board_number` convert between board numbers and their dealer/vulnerability.

In [ ]:
for board in range(1, 17):
    dealer, vuln = bp.board_number_to_dealer_vuln(board)
    print(f'Board {board:2d}: dealer={dealer}  vuln={vuln}')

In [ ]:
# Use real board vulnerabilities when scoring
df = bp.random_deals(16, seed=1)
board_numbers = pd.RangeIndex(1, 17)
vulns = pd.Series([str(bp.board_number_to_dealer_vuln(b)[1]) for b in board_numbers])

bp.add_dds(df, '3N-N', 'score_3nn_real_vuln', vulns)
df['score_3nn_real_vuln']